# Temporal analysis

Temporal analysis counts gaps between distinct calendar days, measures
interval regularity, and checks sortedness using adjacent values. Date
analysis requires at least 30 distinct days.


## Modal cadence and gaps

The pass sorts and deduplicates the observed days, then measures the gaps
between consecutive distinct days. The modal gap (most frequent) is the
column's assumed cadence; the median gap is reported alongside it. Gap
semantics depend on the cadence:

- **daily cadence** (modal 1): missing *weekdays* are counted, and a pure
  weekend bridge is not a gap. Holidays are not modeled — the finding's
  reference says so explicitly (`weekdays`, source `inferred`).
- **weekly cadence** (modal 7): gaps are counted in weeks once more than
  ten days elapse.
- **other cadences**: a gap is flagged when the elapsed time exceeds
  1.5× the modal interval *and* the modal interval plus two days; the
  count is expressed in modal-interval units and marked approximate.

At most 10 gap events are reported, ordered by missing count. Each
`gaps` finding carries its reference — the cadence or weekday rule that
defines "missing" — because an empty interval is only meaningful against
an expectation.

## Irregular intervals

The coefficient of variation (CV) of the day gaps measures variation in
observed spacing. A CV above 0.5, with at least ten gaps, produces an
`intervals` statement containing the modal interval, median interval, and
CV. Irregular spacing can affect resampling and lag features that assume
a regular time index.


## Sortedness

The scan compares adjacent non-null values and requires at least 100
values. It reports strictly increasing or non-decreasing order when those
properties hold. These `ordering` statements describe the relationship
between row order and column values. The pass does not emit a finding for
an unsorted column.


In [ ]:
import datetime as dt
import random

import polars as pl

import onceover

rng = random.Random(51)

# Frame A: a daily snapshot calendar with a two-weekday hole.
base = dt.date(2024, 1, 1)
skip = {dt.date(2024, 2, 8), dt.date(2024, 2, 9)}
business, i = [], 0
while len(business) < 3_000:
    d = base + dt.timedelta(days=i // 10)
    if d not in skip:
        business.append(d)
    i += 1
daily = pl.DataFrame({"row_id": range(len(business)), "snapshot_date": business})

report = onceover.profile(daily)
for f in report.findings:
    if f["category"] in {"gaps", "ordering"}:
        m = f["evidence"][0]["metrics"]
        print(f"[{f['category']:9s}] {f['claim']}")
        if f["category"] == "gaps":
            print(f"            reference={m.get('reference')} cadence_days={m.get('cadence_days')}")

# Frame B: a bursty event log — an uneven clock with silences.
events, day = [], 0
while len(events) < 3_000:
    for _ in range(5):                # five consecutive active days
        day += 1
        events.append(day)
    day += rng.randrange(9, 15)       # then a silence
log_dates = [dt.date(2024, 1, 1) + dt.timedelta(days=d) for d in events]
rng.shuffle(log_dates)                # logs arrive out of order
log = pl.DataFrame({"event_id": range(len(log_dates)), "event_day": log_dates})

log_report = onceover.profile(log)
log_gaps = [f for f in log_report.findings if f["category"] == "gaps"]
print("gap events in the event log:", len(log_gaps))
for f in log_gaps[:2]:
    print(f"[gaps      ] {f['claim']}")
for f in log_report.findings:
    if f["category"] == "intervals":
        print(f"[intervals ] {f['claim']}")

The snapshot calendar is missing two weekdays under the inferred weekday
reference. Its dates are non-decreasing in row order. The event log has
irregular intervals: gaps are one day within bursts and ten or more days
between bursts. Those longer intervals also produce gap events against
the daily modal interval. The report retains the ten largest gap events;
the example prints two of them. The shuffled event-log date column
receives no sortedness finding.

## Datetime columns

Datetime analysis uses day granularity. It does not detect time-of-day
patterns, intraday gaps, or session boundaries
([](../appendix/directions.ipynb)). Gap counts use all distinct days, but
their interpretation depends on the inferred cadence or weekday reference.
